In [1]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path
from collections import namedtuple
import json

project_root = str(ipynbname.path().parent.parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader
from analysis import fit_peak_expbg
from visualization  import hist_quickShow

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used
# Energy emission peaks in MeV
K40_MEV = [1.460]
TL208_MEV = [2.614]
CS137_MEV = [0.661]
ANNIHIL_MEV = 0.511
NA22_MEV = [ANNIHIL_MEV, 1.2745, 1.2745+ANNIHIL_MEV]
CO60_MEV = [(1.1732+1.3325)/2, 1.1732+1.3325] # First value is the average of the two photopeaks. They have the same branching ratio

# colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kAzure, ROOT.kSpring, ROOT.kPink, ROOT.kMagenta, ROOT.kTeal, ROOT.kYellow, ROOT.kViolet, ROOT.kCyan]
# colors = [ROOT.kAzure+9, ROOT.kOrange+1, ROOT.kRed+1, ROOT.kGray+1, ROOT.kViolet-5, ROOT.kRed-5, ROOT.kOrange+8, ROOT.kYellow-5, ROOT.kGray+2, ROOT.kCyan-8]
colors = [ROOT.kAzure-7, ROOT.kOrange+1, ROOT.kRed-7, ROOT.kCyan-5, ROOT.kGreen-5, ROOT.kOrange-4, ROOT.kMagenta-5, ROOT.kRed-9, ROOT.kOrange-7, ROOT.kGray+1]

In [2]:
N_BINS = 5000 # Number of bins for histograms

acq_nrs = ['09',]
acq_nrs.extend([f'{_}' for _ in range(10, 29)])
acq_nrs.pop(18)

files = sorted([list((project_root/'data/260527').glob(f'time_{nr}*.csv'))[0] for nr in acq_nrs])

freqs_kHz = [0.200, 0.400, 0.800, 1.600, 3.200, 6.400, 12.800, 25.600, 52.000, 4.000, 4.500, 4.800, 5.000, 5.500, 5.800, 5.400, 5.300, 5.200, 5.100]

Handler = namedtuple('Handler', ('freq', 'expo', 'data', 'hist', 'led_evts'))

handlers = []
for file, freq in zip(files, freqs_kHz):
    # print(f'{file=}, {freq=}')
    data = np.loadtxt(file, delimiter=',', skiprows=1, usecols=1, dtype=float)

    file_name = file.stem[5:]
    with open(file.parent/f'{file_name}.json') as f:
        meta = json.load(f)
        expo = meta['acquisition']['exposure_sec']

    with open(file.parent/f'{file_name}.csv') as f:
        led_evts = sum(1 for _ in f) - 1

    hist = ROOT.TH1F(f"freq_{freq}", "", N_BINS, 0, data.max())
    hist.Fill(data)
    hist.GetXaxis().SetTitle('Inter-arrival time (s)')
    hist.GetYaxis().SetTitle('Counts')
    
    handlers.append(Handler(freq=freq, expo=expo, data=data, hist=hist, led_evts=led_evts))

handlers

[Handler(freq=0.2, expo=245.36038756370544, data=array([5.300e-05, 1.600e-05, 1.400e-05, ..., 4.962e-03, 4.971e-03,
        4.962e-03], shape=(50000,)), hist=<cppyy.gbl.TH1F object at 0x625e62662bc0>, led_evts=49750),
 Handler(freq=0.4, expo=123.40474534034729, data=array([1.090e-04, 1.700e-05, 1.400e-05, ..., 2.471e-03, 2.461e-03,
        2.471e-03], shape=(50000,)), hist=<cppyy.gbl.TH1F object at 0x625e629dd0b0>, led_evts=49875),
 Handler(freq=0.8, expo=61.082777976989746, data=array([5.400e-05, 1.600e-05, 1.400e-05, ..., 1.220e-03, 1.219e-03,
        1.223e-03], shape=(50000,)), hist=<cppyy.gbl.TH1F object at 0x625e62871b80>, led_evts=49937),
 Handler(freq=1.6, expo=30.61537480354309, data=array([5.10e-05, 1.50e-05, 1.30e-05, ..., 5.89e-04, 5.87e-04, 5.94e-04],
       shape=(50000,)), hist=<cppyy.gbl.TH1F object at 0x625e5c0dabc0>, led_evts=49974),
 Handler(freq=3.2, expo=15.366611957550049, data=array([1.02e-04, 1.60e-05, 1.40e-05, ..., 2.93e-04, 2.91e-04, 2.88e-04],
       shape=(

In [3]:
hist_quickShow([h.hist for h in handlers])

In [4]:
# Frequency vs exposure plot

freqs = np.array([h.freq for h in handlers])
expos = np.array([h.expo for h in handlers])

expoVsFreq = ROOT.TGraph(len(freqs), freqs, expos)

In [5]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 800, 600)

expoVsFreq.SetMarkerColor(colors[0])
expoVsFreq.SetMarkerStyle(20)

expoVsFreq.SetTitle("Exposure Vs frequency")
expoVsFreq.GetXaxis().SetTitle("Frequency (kHz)")
expoVsFreq.GetYaxis().SetTitle("Exposure (s)")

expoVsFreq.Draw("AP")

cv.SetLogx()
cv.SetLogy()
cv.Draw()

In [6]:
fit_eVf = ROOT.TF1('Fit_expoVsFreq_pow', '[0] +[1]*x^[2]', 0, 5.1)
expoVsFreq.Fit(fit_eVf, 'RS')

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =     0.499086
NDf                       =            7
Edm                       =  4.72998e-06
NCalls                    =          175
p0                        =   -0.0775202   +/-   0.193882    
p1                        =      49.3171   +/-   0.287127    
p2                        =    -0.997399   +/-   0.00340889  


In [11]:
# Event detection efficiency

expos_expected = np.array([h.led_evts/(1000*h.freq) for h in handlers])
efficiencies = expos_expected/expos
efficiencies

array([1.01381483, 1.01039469, 1.02191243, 1.02019819, 1.01650986,
       0.50408325, 0.33367811, 0.19508399, 0.09797991, 1.01210906,
       1.01643269, 1.01298979, 1.00528882, 0.5097209 , 0.50963952,
       0.50972381, 0.50972975, 0.7557533 , 1.01275993])

Some efficiencies are larger than 1 because, at low frequencies, there are high-energy events from cosmic rains that migh impinge on the BGO in between LED pulses, so the fraction of events captured is larger than the LED event rate. The number of captured events $N_\text{LED, comp} = N_\text{LED, real} + N_\text{bg}$, where the number of background events is given by $\lambda_\text{bg}\cdot T_\text{acq}$

Background events with energies within the energy window of the detector are indistinguishable from LED events in the inter-arrival times histogram. The compound rate, including LED and background events is

$\eta_\text{comp} = \dfrac{N_\text{LED, real} + \lambda_\text{bg} \cdot T_\text{acq}}{f \cdot T_\text{acq}} = \eta_\text{real} + \frac{\lambda_\text{bg}}{f}$

For $f < f_\text{crit}$, where $\eta_\text{real} = 1$,

$\eta_{\text{comp}} = 1 + \dfrac{\lambda_\text{bg}}{f}$

where $f_\text{crit} = 2.5 \mathrm{kHz}$, the frequency at which there is a discontinuity in the acquisition time

In [23]:
# Correcting for cosmic rays high-energy events
F_CRIT = 2.5 # kHz
mask_low = freqs < F_CRIT
lambda_bg_estimates = (efficiencies[mask_low] - 1) * freqs[mask_low]

lambda_bg = np.mean(lambda_bg_estimates)
lambda_bg_err = np.std(lambda_bg_estimates)

print(f"λ_bg = {lambda_bg:.3f} ± {lambda_bg_err:.3f} Hz")

λ_bg = 0.014 ± 0.012 Hz


# Corrección por fondo en la ventana de energía del LED

## Motivación

La eficiencia calculada como $\eta = N_\text{LED,cap} / (f \cdot T_\text{acq})$ puede superar 1 si eventos de radiación de fondo que caen dentro de la ventana de energía del fotopico del LED son contados como eventos del LED. El modelo completo es:

$$\eta_\text{comp}(f) = \eta_\text{real}(f) + \frac{\lambda_\text{bg}}{f}$$

Para $f < f_\text{crit}$ donde $\eta_\text{real} = 1$, cualquier exceso por encima de 1 es atribuible a fondo:

$$\lambda_\text{bg} = \left(\eta_\text{comp} - 1\right) \cdot f = \text{const.}$$

## Resultado

| Parámetro | Valor |
|---|---|
| $\lambda_\text{bg}$ | $0.014 \pm 0.012$ Hz |
| Significancia | $< 1.2\sigma$ |

El resultado es **compatible con cero**: no hay contaminación de fondo significativa en la ventana de energía del LED.

## Interpretación

La incertidumbre estadística intrínseca de cada punto de eficiencia (con $N \approx 50\,000$ eventos) es:

$$\sigma(\eta) \approx \frac{1}{\sqrt{N}} \approx 0.45\%$$

La señal de fondo estimada a la frecuencia más baja del experimento es:

$$\frac{\lambda_\text{bg}}{f_\text{min}} \approx 0.014\% \ll \sigma(\eta)$$

Los valores $\eta > 1$ observados son por lo tanto **fluctuaciones estadísticas normales**, no un efecto sistemático.

## Conclusión

- No se aplica corrección por fondo.
- Las barras de error en la curva de eficiencia son $\sigma(\eta) \approx \eta / \sqrt{N_\text{LED}}$.
- El fotopico del LED está bien aislado del espectro de fondo, confirmando la correcta selección del umbral de energía.

In [26]:
# Efficiency vs exposure plot
effVsFreq = ROOT.TGraph(len(freqs), freqs, efficiencies)

# Efficiency model based on non-paralyzable behavior
tau_init = 1 / 5.2   # initial estimation of dead-time
f_model = np.logspace(np.log10(freqs.min()), np.log10(freqs.max()), 500)
k_model = np.ceil(tau_init * f_model)           # número de pulsos saltados
eta_model = 1.0 / k_model                       # eficiencia del modelo
T_model   = 50000 * k_model / f_model               # tiempo de adquisición modelo

eff_model = ROOT.TGraph(len(f_model), f_model, eta_model)
expoVsFreq_model = ROOT.TGraph(len(f_model), f_model, T_model)

In [37]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 800, 600)
lg = ROOT.TLegend(0.14, 0.21, 0.41, 0.43)

effVsFreq.SetMarkerColor(colors[0])
effVsFreq.SetMarkerStyle(20)

effVsFreq.SetTitle("Efficiency Vs frequency")
effVsFreq.GetXaxis().SetTitle("Frequency (kHz)")
effVsFreq.GetYaxis().SetTitle("Efficiency (s)")

eff_model.SetLineStyle(2)
eff_model.SetLineColor(colors[2])

effVsFreq.Draw("AP")
eff_model.Draw("C sames")

lg.AddEntry(effVsFreq, "Measured")
lg.AddEntry(eff_model, "Model")

lg.Draw()

cv.SetLogx()
cv.Draw()